In [1]:
from nba_api.stats.endpoints import MatchupsRollup

rollup = MatchupsRollup(
    season="2025-26",
    season_type_playoffs="Regular Season",
    per_mode_simple="Totals",
).matchups_rollup.get_data_frame()

print("Rows:", len(rollup))
print("Columns:", rollup.columns.tolist())

display(rollup["POSITION"].value_counts(dropna=False))

display(
    rollup[
        [
            "DEF_PLAYER_NAME",
            "POSITION",
            "PERCENT_OF_TIME",
            "PARTIAL_POSS",
            "MATCHUP_MIN",
        ]
    ].head(15)
)


Rows: 2326
Columns: ['SEASON_ID', 'POSITION', 'PERCENT_OF_TIME', 'DEF_PLAYER_ID', 'DEF_PLAYER_NAME', 'GP', 'MATCHUP_MIN', 'PARTIAL_POSS', 'PLAYER_PTS', 'TEAM_PTS', 'MATCHUP_AST', 'MATCHUP_TOV', 'MATCHUP_BLK', 'MATCHUP_FGM', 'MATCHUP_FGA', 'MATCHUP_FG_PCT', 'MATCHUP_FG3M', 'MATCHUP_FG3A', 'MATCHUP_FG3_PCT', 'MATCHUP_FTM', 'MATCHUP_FTA', 'SFL']


POSITION
G        582
F        582
TOTAL    582
C        580
Name: count, dtype: int64

,DEF_PLAYER_NAME,POSITION,PERCENT_OF_TIME,PARTIAL_POSS,MATCHUP_MIN
0,LeBron James,G,0.420201,1616.7,315:04
1,LeBron James,F,0.475616,1823.0,356:37
2,LeBron James,C,0.104183,421.6,78:07
3,LeBron James,TOTAL,1.000000,3861.3,749:48
4,Chris Paul,G,0.680626,275.4,53:38
5,Chris Paul,C,0.051607,30.2,4:04
6,Chris Paul,F,0.267766,117.9,21:06
7,Chris Paul,TOTAL,1.000000,423.5,78:48
8,Kyle Lowry,TOTAL,1.000000,238.3,43:48
9,Kyle Lowry,C,0.062785,13.8,2:45


In [3]:

from pathlib import Path
from nba_archetypes.config import RAW_DATA_DIR

raw_path = Path(
    RAW_DATA_DIR / "matchups_rollup.parquet"
)

raw_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

rollup.to_parquet(raw_path, index=False)


In [4]:

import numpy as np
import pandas as pd


POSITIONS = ["G", "F", "C"]


def calculate_matchup_diversity(
    rollup,
    min_possessions=200,
):
    """Measure diversity of recorded G/F/C matchups."""

    required = {
        "DEF_PLAYER_ID",
        "POSITION",
        "PARTIAL_POSS",
    }

    if not required.issubset(rollup.columns):
        raise ValueError("Missing matchup columns.")

    data = rollup.copy()

    data["POSITION"] = (
        data["POSITION"].astype(str).str.strip().str.upper()
    )

    data["PARTIAL_POSS"] = pd.to_numeric(
        data["PARTIAL_POSS"],
        errors="raise",
    )

    valid_positions = set(POSITIONS + ["TOTAL"])

    if not data["POSITION"].isin(valid_positions).all():
        raise ValueError("Unexpected position category.")

    if (
        data["PARTIAL_POSS"].isna().any()
        or (data["PARTIAL_POSS"] < 0).any()
    ):
        raise ValueError("Invalid matchup possessions.")

    if data.duplicated(
        ["DEF_PLAYER_ID", "POSITION"]
    ).any():
        raise ValueError("Duplicate defender-position rows.")

    # Keep the three offensive position categories.
    position_rows = data[
        data["POSITION"].isin(POSITIONS)
    ]

    counts = position_rows.pivot(
        index="DEF_PLAYER_ID",
        columns="POSITION",
        values="PARTIAL_POSS",
    )

    counts = counts.reindex(
        columns=POSITIONS
    ).fillna(0)

    # Total recorded matchup possessions.
    volume = counts.sum(axis=1)

    # Share of matchups against each position.
    shares = counts.div(
        volume.replace(0, np.nan),
        axis=0,
    ).fillna(0)

    p = shares.to_numpy()

    # Handle zero shares without taking log(0).
    safe_p = np.where(p > 0, p, 1.0)

    entropy = -(p * np.log(safe_p)).sum(axis=1)
    diversity = 100 * entropy / np.log(3)

    result = shares.rename(
        columns={
            "G": "MATCHUP_SHARE_G",
            "F": "MATCHUP_SHARE_F",
            "C": "MATCHUP_SHARE_C",
        }
    ).copy()

    result["MATCHUP_POSSESSIONS"] = volume
    result["MATCHUP_DIVERSITY_RAW"] = diversity

    # Compare G/F/C totals against the endpoint's
    # supplied TOTAL row as a data-quality check.
    total_rows = data[
        data["POSITION"] == "TOTAL"
    ].set_index("DEF_PLAYER_ID")["PARTIAL_POSS"]

    result["ROLLUP_TOTAL_POSS"] = total_rows

    result["ROLLUP_COVERAGE"] = (
        result["MATCHUP_POSSESSIONS"]
        / result["ROLLUP_TOTAL_POSS"].replace(0, np.nan)
    )

    # Provisional quality thresholds.
    result["VALID_FOR_MODEL"] = (
        (result["MATCHUP_POSSESSIONS"] >= min_possessions)
        & result["ROLLUP_COVERAGE"].between(0.95, 1.05)
    )

    # Missing means insufficient or unverified evidence,
    # NOT zero defensive versatility.
    result["POSITIONAL_MATCHUP_DIVERSITY"] = (
        result["MATCHUP_DIVERSITY_RAW"].where(
            result["VALID_FOR_MODEL"]
        )
    )

    return result.reset_index()


In [5]:
from nba_archetypes.feature_engineering import (
    calculate_matchup_diversity,
)

diversity = calculate_matchup_diversity(rollup)

print("Total defenders:", len(diversity))
print("Valid defenders:", diversity["VALID_FOR_MODEL"].sum())

print("\nCoverage statistics:")
display(
    diversity["ROLLUP_COVERAGE"].describe()
)

print("\nMatchup possession statistics:")
display(
    diversity["MATCHUP_POSSESSIONS"].describe()
)

print("\nDiversity distribution:")
display(
    diversity["POSITIONAL_MATCHUP_DIVERSITY"].describe()
)

print("\nHighest matchup diversity:")
display(
    diversity.nlargest(
        10, "POSITIONAL_MATCHUP_DIVERSITY"
    )
)

print("\nLowest valid matchup diversity:")
display(
    diversity[
        diversity["VALID_FOR_MODEL"]
    ].nsmallest(
        10, "POSITIONAL_MATCHUP_DIVERSITY"
    )
)


Total defenders: 582
Valid defenders: 503

Coverage statistics:


count    5.820000e+02
mean     1.000000e+00
std      1.027867e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: ROLLUP_COVERAGE, dtype: float64


Matchup possession statistics:


count     582.000000
mean     2006.947766
std      1530.139541
min         4.000000
25%       571.325000
50%      1865.350000
75%      3170.725000
max      5668.400000
Name: MATCHUP_POSSESSIONS, dtype: float64


Diversity distribution:


count    503.000000
mean      82.598431
std       10.803001
min       56.658883
25%       74.653312
50%       82.056754
75%       92.344844
max       99.832465
Name: POSITIONAL_MATCHUP_DIVERSITY, dtype: float64


Highest matchup diversity:


POSITION,DEF_PLAYER_ID,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,MATCHUP_DIVERSITY_RAW,ROLLUP_TOTAL_POSS,ROLLUP_COVERAGE,VALID_FOR_MODEL,POSITIONAL_MATCHUP_DIVERSITY
330,1631131,0.305666,0.352844,0.341490,907.2,99.832465,907.2,1.0,True,99.832465
232,1630264,0.318142,0.377301,0.304557,1928.7,99.598493,1928.7,1.0,True,99.598493
414,1641765,0.329635,0.378665,0.291700,2019.2,99.482286,2019.2,1.0,True,99.482286
265,1630578,0.304329,0.384647,0.311024,4555.6,99.469757,4555.6,1.0,True,99.469757
484,1642366,0.282642,0.351435,0.365923,2222.6,99.445426,2222.6,1.0,True,99.445426
4,201143,0.288625,0.380873,0.330502,1860.2,99.418027,1860.2,1.0,True,99.418027
530,1642867,0.288682,0.387805,0.323513,2382.9,99.316746,2382.9,1.0,True,99.316746
223,1630214,0.282457,0.334072,0.383471,315.8,99.299831,315.8,1.0,True,99.299831
37,203486,0.324588,0.287065,0.388347,327.8,99.290435,327.8,1.0,True,99.290435
353,1631218,0.274984,0.360570,0.364446,954.6,99.278498,954.6,1.0,True,99.278498



Lowest valid matchup diversity:


POSITION,DEF_PLAYER_ID,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,MATCHUP_DIVERSITY_RAW,ROLLUP_TOTAL_POSS,ROLLUP_COVERAGE,VALID_FOR_MODEL,POSITIONAL_MATCHUP_DIVERSITY
552,1642918,0.798088,0.145698,0.056214,261.5,56.658883,261.5,1.0,True,56.658883
524,1642859,0.775147,0.189298,0.035555,1119.4,57.448861,1119.4,1.0,True,57.448861
63,204456,0.767718,0.196714,0.035567,1917.5,58.387725,1917.5,1.0,True,58.387725
158,1629162,0.756486,0.206022,0.037493,528.1,60.047870,528.1,1.0,True,60.047870
505,1642484,0.760156,0.198492,0.041352,411.1,60.180937,411.1,1.0,True,60.180937
464,1642278,0.754361,0.205130,0.040509,1352.8,60.755860,1352.8,1.0,True,60.755860
359,1631245,0.757701,0.198704,0.043595,1821.3,60.795431,1821.3,1.0,True,60.795431
540,1642880,0.732297,0.234361,0.033341,1235.7,62.040645,1235.7,1.0,True,62.040645
471,1642347,0.755459,0.191559,0.052982,3544.6,62.266416,3544.6,1.0,True,62.266416
245,1630540,0.751372,0.199466,0.049162,2058.5,62.301424,2058.5,1.0,True,62.301424


In [8]:

# Join the new defensive feature to our eligible players.
from nba_archetypes.config import INTERIM_DATA_DIR
players = pd.read_parquet(
    INTERIM_DATA_DIR / "eligible_players.parquet"
)
eligible_diversity = players[
    ["PLAYER_ID", "PLAYER_NAME"]
].merge(
    diversity,
    left_on="PLAYER_ID",
    right_on="DEF_PLAYER_ID",
    how="left",
    validate="one_to_one",
)

valid = eligible_diversity[
    "POSITIONAL_MATCHUP_DIVERSITY"
].notna()

print("Eligible players:", len(eligible_diversity))
print("Eligible players with matchup data:",
      eligible_diversity["MATCHUP_POSSESSIONS"].notna().sum())
print("Eligible players with valid diversity:", valid.sum())
print("Missing or insufficient data:", (~valid).sum())

# See how the exposure threshold affects eligibility.
print("\nCoverage at different possession thresholds:")

for threshold in [200, 500, 1000]:
    count = eligible_diversity[
        "MATCHUP_POSSESSIONS"
    ].ge(threshold).sum()

    print(f"{threshold} possessions: {count} players")

# Inspect recognizable NBA players at each extreme.
columns = [
    "PLAYER_NAME",
    "MATCHUP_SHARE_G",
    "MATCHUP_SHARE_F",
    "MATCHUP_SHARE_C",
    "MATCHUP_POSSESSIONS",
    "POSITIONAL_MATCHUP_DIVERSITY",
]

print("\nHighest diversity — eligible players:")
display(
    eligible_diversity.loc[valid]
    .nlargest(10, "POSITIONAL_MATCHUP_DIVERSITY")[columns]
    .round(3)
)

print("\nLowest diversity — eligible players:")
display(
    eligible_diversity.loc[valid]
    .nsmallest(10, "POSITIONAL_MATCHUP_DIVERSITY")[columns]
    .round(3)
)


Eligible players: 450
Eligible players with matchup data: 450
Eligible players with valid diversity: 450
Missing or insufficient data: 0

Coverage at different possession thresholds:
200 possessions: 450 players
500 possessions: 446 players
1000 possessions: 378 players

Highest diversity — eligible players:


,PLAYER_NAME,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,POSITIONAL_MATCHUP_DIVERSITY
349,Oscar Tshiebwe,0.306,0.353,0.341,907.2,99.832
23,Anthony Gill,0.318,0.377,0.305,1928.7,99.598
347,Olivier-Maxence Prosper,0.330,0.379,0.292,2019.2,99.482
12,Alperen Sengun,0.304,0.385,0.311,4555.6,99.470
367,Quinten Post,0.283,0.351,0.366,2222.6,99.445
9,Al Horford,0.289,0.381,0.331,1860.2,99.418
78,Collin Murray-Boyles,0.289,0.388,0.324,2382.9,99.317
416,Trayce Jackson-Davis,0.275,0.361,0.364,954.6,99.278
387,Sandro Mamukelashvili,0.296,0.393,0.311,3399.6,99.268
373,Robert Williams III,0.278,0.385,0.337,1962.4,99.198



Lowest diversity — eligible players:


,PLAYER_NAME,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,POSITIONAL_MATCHUP_DIVERSITY
192,Jase Richardson,0.775,0.189,0.036,1119.4,57.449
402,T.J. McConnell,0.768,0.197,0.036,1917.5,58.388
224,Jordan McLaughlin,0.756,0.206,0.037,528.1,60.048
427,Tyler Kolek,0.754,0.205,0.041,1352.8,60.756
366,Quenton Jackson,0.758,0.199,0.044,1821.3,60.795
244,Kam Jones,0.732,0.234,0.033,1235.7,62.041
182,Jamal Shead,0.755,0.192,0.053,3544.6,62.266
317,Miles McBride,0.751,0.199,0.049,2058.5,62.301
430,Tyrese Proctor,0.714,0.260,0.026,1046.9,62.335
209,Jevon Carter,0.741,0.215,0.043,1699.9,62.713


In [10]:
# Keep the three position categories.
time_data = rollup[
    rollup["POSITION"].isin(["G", "F", "C"])
].copy()

time_data["PERCENT_OF_TIME"] = pd.to_numeric(
    time_data["PERCENT_OF_TIME"]
)

# One row per defender, one column per position.
time_shares = time_data.pivot(
    index="DEF_PLAYER_ID",
    columns="POSITION",
    values="PERCENT_OF_TIME",
)

time_shares = (
    time_shares.reindex(columns=["G", "F", "C"])
    .fillna(0)
)

# Check the source percentages.
total_share = time_shares.sum(axis=1)

print(
    "Largest deviation from 1:",
    (total_share - 1).abs().max()
)

if not np.allclose(total_share, 1, atol=0.001):
    raise ValueError("Matchup time shares do not sum to 1.")

# Normalize minor rounding differences.
time_shares = time_shares.div(
    total_share, axis=0
)

# Shannon entropy, normalized to 0–100.
p = time_shares.to_numpy()

safe_p = np.where(p > 0, p, 1.0)

time_entropy = (
    -np.sum(p * np.log(safe_p), axis=1)
    / np.log(3)
    * 100
)

time_diversity = pd.DataFrame({
    "DEF_PLAYER_ID": time_shares.index,
    "MATCHUP_DIVERSITY_TIME": time_entropy,
}).reset_index(drop=True)

display(time_diversity.head())

comparison = (
    players[["PLAYER_ID", "PLAYER_NAME"]]
    .merge(
        diversity[
            [
                "DEF_PLAYER_ID",
                "POSITIONAL_MATCHUP_DIVERSITY",
            ]
        ],
        left_on="PLAYER_ID",
        right_on="DEF_PLAYER_ID",
        validate="one_to_one",
    )
    .merge(
        time_diversity,
        on="DEF_PLAYER_ID",
        validate="one_to_one",
    )
)

assert len(comparison) == 450

pos_col = "POSITIONAL_MATCHUP_DIVERSITY"
time_col = "MATCHUP_DIVERSITY_TIME"

# Compare the numerical scores.
comparison["ABS_DIFFERENCE"] = (
    comparison[pos_col] - comparison[time_col]
).abs()

# Compare player rankings (highest diversity first).
comparison["POSSESSION_RANK"] = (
    comparison[pos_col].rank(
        ascending=False, method="min"
    )
)

comparison["TIME_RANK"] = (
    comparison[time_col].rank(
        ascending=False, method="min"
    )
)

comparison["RANK_CHANGE"] = (
    comparison["POSSESSION_RANK"]
    - comparison["TIME_RANK"]
).abs()

print("Pearson correlation:",
      round(comparison[pos_col].corr(comparison[time_col]), 4))

print("Spearman rank correlation:",
      round(
          comparison[pos_col].corr(
              comparison[time_col], method="spearman"
          ), 4
      ))

print("Mean absolute score difference:",
      round(comparison["ABS_DIFFERENCE"].mean(), 3))

print("Median rank change:",
      comparison["RANK_CHANGE"].median())

print("\nPlayers with largest ranking changes:")

display(
    comparison.nlargest(15, "RANK_CHANGE")[
        [
            "PLAYER_NAME",
            pos_col,
            time_col,
            "POSSESSION_RANK",
            "TIME_RANK",
            "RANK_CHANGE",
        ]
    ].round(2)
)

Largest deviation from 1: 1.0014211682118912e-13


,DEF_PLAYER_ID,MATCHUP_DIVERSITY_TIME
0,2544,86.781880
1,101108,69.874924
2,200768,79.552685
3,201142,83.437061
4,201143,98.173528


Pearson correlation: 0.9913
Spearman rank correlation: 0.9929
Mean absolute score difference: 2.057
Median rank change: 9.0

Players with largest ranking changes:


,PLAYER_NAME,POSITIONAL_MATCHUP_DIVERSITY,MATCHUP_DIVERSITY_TIME,POSSESSION_RANK,TIME_RANK,RANK_CHANGE
445,Zach Edey,89.68,81.46,131.0,215.0,84.0
68,Chaney Johnson,95.09,96.79,93.0,37.0,56.0
290,Lindy Waters III,74.36,74.77,345.0,302.0,43.0
71,Christian Braun,77.80,73.49,283.0,325.0,42.0
38,Bobby Portis Jr.,96.61,97.44,68.0,28.0,40.0
337,Nikola Jović,95.02,95.68,95.0,55.0,40.0
415,Trae Young,73.44,65.36,356.0,395.0,39.0
129,Egor Dëmin,74.55,74.41,342.0,304.0,38.0
448,Ziaire Williams,76.06,76.42,315.0,277.0,38.0
72,Christian Koloko,96.75,92.36,63.0,100.0,37.0


In [14]:
from nba_archetypes.config import PROCESSED_DATA_DIR
features = pd.read_parquet(
    PROCESSED_DATA_DIR / "player_style_features.parquet"
)

defense_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_DEFENSE_SCORE",
    "DEFENSIVE_REBOUNDING_SCORE",
]

defense_corr = features[defense_cols].corr()

display(defense_corr.round(3))

print(
    "Interior Defense vs Defensive Rebounding:",
    round(
        defense_corr.loc[
            "INTERIOR_DEFENSE_SCORE",
            "DEFENSIVE_REBOUNDING_SCORE",
        ],
        3,
    )
)


defense_experiment = features[
    ["PLAYER_ID", "PLAYER_NAME"] + defense_cols
].copy()

# Combine the two previously standardized dimensions.
interior_combined = (
    defense_experiment["INTERIOR_DEFENSE_SCORE"]
    + defense_experiment["DEFENSIVE_REBOUNDING_SCORE"]
) / 2

# Standardize the combined dimension.
defense_experiment["INTERIOR_PRESENCE_SCORE"] = (
    interior_combined - interior_combined.mean()
) / interior_combined.std(ddof=0)

display(
    defense_experiment[
        [
            "INTERIOR_DEFENSE_SCORE",
            "DEFENSIVE_REBOUNDING_SCORE",
            "INTERIOR_PRESENCE_SCORE",
        ]
    ].corr().round(3)
)


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_DEFENSE_SCORE,DEFENSIVE_REBOUNDING_SCORE
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.078,-0.111
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.765,-0.538
INTERIOR_DEFENSE_SCORE,-0.078,-0.765,1.000,0.668
DEFENSIVE_REBOUNDING_SCORE,-0.111,-0.538,0.668,1.000


Interior Defense vs Defensive Rebounding: 0.668


,INTERIOR_DEFENSE_SCORE,DEFENSIVE_REBOUNDING_SCORE,INTERIOR_PRESENCE_SCORE
INTERIOR_DEFENSE_SCORE,1.000,0.668,0.913
DEFENSIVE_REBOUNDING_SCORE,0.668,1.000,0.913
INTERIOR_PRESENCE_SCORE,0.913,0.913,1.000


In [15]:

# Start with the eligible-player defensive scores.
defense_experiment = features[
    [
        "PLAYER_ID",
        "PLAYER_NAME",
        "DEFENSIVE_DISRUPTION_SCORE",
        "PERIMETER_DEFENSE_SCORE",
        "INTERIOR_DEFENSE_SCORE",
        "DEFENSIVE_REBOUNDING_SCORE",
    ]
].copy()

# Merge time-weighted matchup diversity.
defense_experiment = defense_experiment.merge(
    time_diversity,
    left_on="PLAYER_ID",
    right_on="DEF_PLAYER_ID",
    how="left",
    validate="one_to_one",
)

assert len(defense_experiment) == 450
assert defense_experiment["MATCHUP_DIVERSITY_TIME"].notna().all()

# Combine Interior Defense and Defensive Rebounding.
interior = defense_experiment[
    ["INTERIOR_DEFENSE_SCORE", "DEFENSIVE_REBOUNDING_SCORE"]
].mean(axis=1)

defense_experiment["INTERIOR_PRESENCE_SCORE"] = (
    (interior - interior.mean())
    / interior.std(ddof=0)
)

# Standardize matchup diversity using eligible players.
matchup = defense_experiment["MATCHUP_DIVERSITY_TIME"]

defense_experiment["MATCHUP_DIVERSITY_SCORE"] = (
    (matchup - matchup.mean())
    / matchup.std(ddof=0)
)

# Candidate four-dimensional representation.
defense_style_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_PRESENCE_SCORE",
    "MATCHUP_DIVERSITY_SCORE",
]

print("Correlation matrix:")
display(
    defense_experiment[defense_style_cols]
    .corr()
    .round(3)
)

print("\nDimension distributions:")
display(
    defense_experiment[defense_style_cols]
    .describe()
    .round(3)
)


Correlation matrix:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,MATCHUP_DIVERSITY_SCORE
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.103,-0.131
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.714,-0.566
INTERIOR_PRESENCE_SCORE,-0.103,-0.714,1.000,0.777
MATCHUP_DIVERSITY_SCORE,-0.131,-0.566,0.777,1.000



Dimension distributions:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,MATCHUP_DIVERSITY_SCORE
count,450.000,450.000,450.000,450.000
mean,-0.000,0.000,-0.000,0.000
std,1.001,1.001,1.001,1.001
min,-2.792,-2.358,-2.091,-2.475
25%,-0.640,-0.709,-0.775,-0.704
50%,-0.041,0.001,-0.236,0.010
75%,0.636,0.622,0.552,0.897
max,3.649,2.652,3.090,1.624


In [16]:

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
)

# Model A: Three core defensive dimensions.
core_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_PRESENCE_SCORE",
]

# Model B: Add positional matchup diversity.
extended_cols = core_cols + [
    "MATCHUP_DIVERSITY_SCORE"
]

X_core = defense_experiment[core_cols].to_numpy()
X_extended = defense_experiment[extended_cols].to_numpy()

defense_models = {}
results = []

for k in range(3, 7):

    core = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )

    extended = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )

    core_labels = core.fit_predict(X_core)
    extended_labels = extended.fit_predict(X_extended)

    defense_models[("Core", k)] = core
    defense_models[("Extended", k)] = extended

    core_sizes = pd.Series(core_labels).value_counts()
    extended_sizes = pd.Series(extended_labels).value_counts()

    results.append({
        "k": k,
        "Core Silhouette": silhouette_score(
            X_core, core_labels
        ),
        "Extended Silhouette": silhouette_score(
            X_extended, extended_labels
        ),
        "ARI": adjusted_rand_score(
            core_labels, extended_labels
        ),
        "Core Min": core_sizes.min(),
        "Core Max": core_sizes.max(),
        "Extended Min": extended_sizes.min(),
        "Extended Max": extended_sizes.max(),
    })

defense_results = pd.DataFrame(results)

display(defense_results.round(3))


,k,Core Silhouette,Extended Silhouette,ARI,Core Min,Core Max,Extended Min,Extended Max
0,3,0.292,0.269,0.700,116,184,138,170
1,4,0.286,0.253,0.592,96,125,100,129
2,5,0.281,0.231,0.454,63,123,74,117
3,6,0.278,0.240,0.504,53,121,50,117


In [17]:

# Add time-weighted G/F/C matchup shares.
role_experiment = defense_experiment.merge(
    time_shares.add_prefix("TIME_SHARE_"),
    left_on="PLAYER_ID",
    right_index=True,
    validate="one_to_one",
)

# Two contrasts preserve the information in all three shares.
role_experiment["GUARD_CENTER_BALANCE"] = (
    role_experiment["TIME_SHARE_G"]
    - role_experiment["TIME_SHARE_C"]
)

role_experiment["FORWARD_FOCUS"] = (
    role_experiment["TIME_SHARE_F"]
    - (
        role_experiment["TIME_SHARE_G"]
        + role_experiment["TIME_SHARE_C"]
    ) / 2
)

assignment_cols = [
    "GUARD_CENTER_BALANCE",
    "FORWARD_FOCUS",
]

# Standardize the two assignment dimensions.
assignment = role_experiment[assignment_cols]

assignment_scaled = (
    (assignment - assignment.mean())
    / assignment.std(ddof=0)
)

# Balance the two assignment dimensions as a group.
assignment_scaled = assignment_scaled / np.sqrt(2)

# Combine with the three core dimensions.
X_role = np.column_stack([
    role_experiment[core_cols].to_numpy(),
    assignment_scaled.to_numpy(),
])

print("Shape:", X_role.shape)

display(
    role_experiment[
        core_cols + assignment_cols
    ].corr().round(3)
)

k = 4

role_model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20,
)

role_labels = role_model.fit_predict(X_role)

core_labels = defense_models[("Core", k)].labels_

print(
    "Core vs. assignment-aware ARI:",
    round(adjusted_rand_score(core_labels, role_labels), 3)
)

print(
    "Assignment-aware silhouette:",
    round(silhouette_score(X_role, role_labels), 3)
)

role_experiment["ROLE_CLUSTER"] = role_labels

print("\nCluster sizes:")
display(
    role_experiment["ROLE_CLUSTER"]
    .value_counts()
    .sort_index()
)

print("\nDefensive profiles:")
display(
    role_experiment.groupby("ROLE_CLUSTER")[
        core_cols + [
            "TIME_SHARE_G",
            "TIME_SHARE_F",
            "TIME_SHARE_C",
        ]
    ].mean().round(3)
)


Shape: (450, 5)


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,GUARD_CENTER_BALANCE,FORWARD_FOCUS
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.103,0.181,-0.173
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.714,0.657,-0.221
INTERIOR_PRESENCE_SCORE,-0.103,-0.714,1.000,-0.880,0.357
GUARD_CENTER_BALANCE,0.181,0.657,-0.880,1.000,-0.414
FORWARD_FOCUS,-0.173,-0.221,0.357,-0.414,1.000


Core vs. assignment-aware ARI: 0.311
Assignment-aware silhouette: 0.253

Cluster sizes:


ROLE_CLUSTER
0     95
1    149
2     86
3    120
Name: count, dtype: int64


Defensive profiles:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,TIME_SHARE_G,TIME_SHARE_F,TIME_SHARE_C
ROLE_CLUSTER,,,,,,
0,1.252,0.700,-0.506,0.617,0.313,0.069
1,-0.309,-0.258,0.088,0.450,0.436,0.114
2,-0.273,-1.249,1.626,0.208,0.361,0.431
3,-0.412,0.662,-0.874,0.661,0.290,0.049
